# Join entropy-split train groups with corrected-answer distillation

For every `group{i}_train.parquet` under `single_token_entropy/mmlu/{model}/`, attach
`distill_reasoning` and `distill_answer` from the corrected-answer distillation parquet
(joined on `question_id`) and write **two truncation variants**:

- **Variant 1 (head):** keep the first 24,000 chars -> `group{i}_train_corrected_answer.parquet`
- **Variant 2 (middle drop):** keep head + tail (24,000 total), dropping the middle ->
  `group{i}_train_corrected_answer_middle_truncated.parquet`.
  Tail size = `len(corrected_reasoning)` when non-empty, else a fixed 6,000 (head 18,000).

In [1]:
from pathlib import Path

import pandas as pd

SPLITS_ROOT = Path("../../data/out/splits/single_token_entropy/mmlu")
DISTILL_PATH = Path(
    "../../data/out/distillation/mmlu_corrected_answer_deepseek_v4_pro_and_others.parquet"
)
MODELS = ["llama_3b", "phi4mini", "qwen_3b"]
N_GROUPS = 6
ID_COL = "question_id"
LOOKUP_COLS = ["distill_reasoning", "distill_answer", "corrected_reasoning"]
MAX_REASONING_CHARS = 24000
DEFAULT_TAIL_CHARS = 6000  # tail kept when corrected_reasoning is empty (head 18000 + tail 6000)

In [2]:
distill_df = pd.read_parquet(DISTILL_PATH)
distill_lookup = distill_df[[ID_COL, *LOOKUP_COLS]]
assert distill_lookup[ID_COL].is_unique, "question_id must be unique for a 1:1 join"
print(f"Loaded {len(distill_lookup)} distillation rows")

Loaded 12032 distillation rows


In [3]:
def middle_truncate(
    reasoning,
    corrected,
    max_chars=MAX_REASONING_CHARS,
    default_tail=DEFAULT_TAIL_CHARS,
):
    """Drop the middle of `reasoning`, keeping a head + tail of `max_chars` total.

    Tail length equals len(corrected_reasoning) when it is non-empty, otherwise
    `default_tail`. NaN / already-short reasoning is returned unchanged.
    """
    if not isinstance(reasoning, str) or len(reasoning) <= max_chars:
        return reasoning
    has_corrected = isinstance(corrected, str) and corrected.strip() != ""
    tail = min(len(corrected), max_chars) if has_corrected else default_tail
    head = max_chars - tail
    if head <= 0:  # corrected_reasoning >= budget -> keep last max_chars
        return reasoning[-max_chars:]
    return reasoning[:head] + reasoning[-tail:]

In [4]:
for model in MODELS:
    model_dir = SPLITS_ROOT / model
    for i in range(N_GROUPS):
        group_df = pd.read_parquet(model_dir / f"group{i}_train.parquet")
        merged = group_df.merge(distill_lookup, on=ID_COL, how="left")
        assert len(merged) == len(group_df), "row count changed — join key not 1:1"
        missing = int(merged["distill_reasoning"].isna().sum())
        too_long = int((merged["distill_reasoning"].str.len() > MAX_REASONING_CHARS).sum())

        # Variant 1: head truncation (first MAX_REASONING_CHARS chars)
        v1 = merged.drop(columns=["corrected_reasoning"]).copy()
        v1["distill_reasoning"] = v1["distill_reasoning"].str.slice(0, MAX_REASONING_CHARS)
        v1.to_parquet(
            str(model_dir / f"group{i}_train_corrected_answer.parquet"), index=False
        )

        # Variant 2: middle drop (head + tail, tail sized by corrected_reasoning)
        v2 = merged.copy()
        v2["distill_reasoning"] = [
            middle_truncate(r, c)
            for r, c in zip(v2["distill_reasoning"], v2["corrected_reasoning"])
        ]
        v2 = v2.drop(columns=["corrected_reasoning"])
        v2.to_parquet(
            str(model_dir / f"group{i}_train_corrected_answer_middle_truncated.parquet"),
            index=False,
        )

        print(
            f"{model}/group{i}: {len(merged)} rows | missing distill={missing} | "
            f"truncated={too_long}"
        )

llama_3b/group0: 1604 rows | missing distill=0 | truncated=48
llama_3b/group1: 1604 rows | missing distill=0 | truncated=98


llama_3b/group2: 1604 rows | missing distill=0 | truncated=103


llama_3b/group3: 1604 rows | missing distill=0 | truncated=124
llama_3b/group4: 1604 rows | missing distill=0 | truncated=140


llama_3b/group5: 1604 rows | missing distill=0 | truncated=226


phi4mini/group0: 1604 rows | missing distill=0 | truncated=16
phi4mini/group1: 1604 rows | missing distill=0 | truncated=83
phi4mini/group2: 1604 rows | missing distill=0 | truncated=78


...

phi4mini/group4: 1604 rows | missing distill=0 | truncated=190
phi4mini/group5: 1604 rows | missing distill=0 | truncated=247


qwen_3b/group0: 1604 rows | missing distill=0 | truncated=34
qwen_3b/group1: 1604 rows | missing distill=0 | truncated=96


qwen_3b/group2: 1604 rows | missing distill=0 | truncated=120
qwen_3b/group3: 1604 rows | missing distill=0 | truncated=144


qwen_3b/group4: 1604 rows | missing distill=0 | truncated=144


qwen_3b/group5: 1604 rows | missing distill=0 | truncated=209


## Verification

In [5]:
base = SPLITS_ROOT / "qwen_3b"
v1 = pd.read_parquet(base / "group0_train_corrected_answer.parquet")
v2 = pd.read_parquet(base / "group0_train_corrected_answer_middle_truncated.parquet")

assert list(v1.columns) == list(v2.columns)
assert {"distill_reasoning", "distill_answer"}.issubset(v1.columns)
assert "corrected_reasoning" not in v1.columns
assert v1["distill_reasoning"].str.len().max() <= MAX_REASONING_CHARS
assert v2["distill_reasoning"].str.len().max() <= MAX_REASONING_CHARS

# On the longest-reasoning row the two variants keep different tails.
long = v1["distill_reasoning"].str.len().idxmax()
assert v1.loc[long, "distill_reasoning"][-50:] != v2.loc[long, "distill_reasoning"][-50:]

print("columns:", list(v1.columns))
print("OK")

columns: ['src', 'answer', 'options', 'category', 'question', 'cot_content', 'question_id', 'answer_index', 'total_tokens', 'meta_cluster', 'base_cluster', 'model_answer', 'model_answer_correct', 'entropy_value', 'distill_reasoning', 'distill_answer']
OK
